# 11 — Churn Deep Dive

**Purpose:** Understand why customers are churning and who to prioritise for re-engagement.  
**Key question:** What are the top reasons for churn and what does the at-risk population look like?  

**Tables used:** `marts.mart_churn_explained`, `marts.mart_churn_risk`, `marts.mart_spend_momentum`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. Churn reasons breakdown
What are the most common reasons customers are flagged as at-risk?

In [ ]:
reasons = q(f"""
    SELECT reason_1 AS reason, COUNT(*) AS customers
    FROM `{PROJECT}.marts.mart_churn_explained`
    GROUP BY reason_1
    ORDER BY customers DESC
""")

if not reasons.empty:
    fig = px.bar(reasons, x='customers', y='reason', orientation='h',
                 color='customers', color_continuous_scale='Reds',
                 title='Primary churn reasons (Critical + High risk customers)',
                 labels={'customers': 'Number of customers', 'reason': ''})
    fig.update_layout(yaxis=dict(autorange='reversed'), height=400, showlegend=False)
    fig.show()
    display(reasons)
else:
    print('No churn explanation data. Run mart_churn_explained SQL first.')

## 2. At-risk customer profile
What do the high-risk customers look like?

In [ ]:
profile = q(f"""
    SELECT
           COUNT(*) AS total_at_risk,
           ROUND(AVG(churn_probability) * 100, 1) AS avg_churn_pct,
           ROUND(AVG(total_spend), 0) AS avg_spend,
           ROUND(SUM(total_spend), 0) AS total_spend_at_risk,
           ROUND(AVG(days_since_last), 0) AS avg_days_inactive,
           ROUND(AVG(txns_last_3m), 1) AS avg_recent_txns,
           ROUND(AVG(txns_prev_3m), 1) AS avg_prior_txns
    FROM `{PROJECT}.marts.mart_churn_explained`
""")

if not profile.empty:
    p = profile.iloc[0]
    print(f'At-risk customer profile (Critical + High):\n')
    print(f'  Total customers:      {int(p["total_at_risk"]):,}')
    print(f'  Avg churn probability: {p["avg_churn_pct"]}%')
    print(f'  Avg historical spend:  R{p["avg_spend"]:,.0f}')
    print(f'  Total spend at risk:   R{p["total_spend_at_risk"]:,.0f}')
    print(f'  Avg days since last:   {p["avg_days_inactive"]:.0f} days')
    print(f'  Txns last 3m vs prior: {p["avg_recent_txns"]:.1f} vs {p["avg_prior_txns"]:.1f}')

## 3. Spend momentum distribution
How many customers are accelerating vs declining?

In [ ]:
momentum = q(f"""
    SELECT momentum_status,
           COUNT(*) AS customers,
           ROUND(AVG(total_spend_12m), 0) AS avg_spend,
           ROUND(AVG(spend_change_pct), 1) AS avg_spend_change
    FROM `{PROJECT}.marts.mart_spend_momentum`
    GROUP BY momentum_status
    ORDER BY avg_spend_change DESC
""")

if not momentum.empty:
    mom_colors = {'Accelerating': '#4CAF50', 'Steady': '#2196f3', 'Slowing': '#FF9800',
                  'Declining': '#f44336', 'New': '#9C27B0'}
    colors = [mom_colors.get(s, '#607D8B') for s in momentum['momentum_status']]

    fig = px.pie(momentum, values='customers', names='momentum_status',
                 color='momentum_status', color_discrete_map=mom_colors,
                 title='Customer spend momentum distribution')
    fig.show()
    display(momentum)

## 4. High-urgency customers
High spend + declining momentum = most urgent to act on.

In [ ]:
urgent = q(f"""
    SELECT momentum_status,
           COUNT(*) AS customers,
           ROUND(AVG(total_spend_12m), 0) AS avg_spend,
           ROUND(AVG(urgency_score), 2) AS avg_urgency,
           ROUND(SUM(total_spend_12m), 0) AS total_spend
    FROM `{PROJECT}.marts.mart_spend_momentum`
    WHERE momentum_status IN ('Declining', 'Slowing')
    GROUP BY momentum_status
""")

if not urgent.empty:
    display(urgent)
    total = urgent['total_spend'].sum()
    custs = urgent['customers'].sum()
    print(f'\n{custs:,} customers are slowing or declining')
    print(f'Their combined 12-month spend: R{total:,.0f}')

## 5. Momentum vs churn alignment
Do declining customers actually have higher churn scores? They should.

In [ ]:
alignment = q(f"""
    SELECT sm.momentum_status,
           ROUND(AVG(cr.churn_probability) * 100, 1) AS avg_churn_pct,
           COUNT(*) AS customers
    FROM `{PROJECT}.marts.mart_spend_momentum` sm
    JOIN `{PROJECT}.marts.mart_churn_risk` cr ON sm.UNIQUE_ID = cr.UNIQUE_ID
    GROUP BY sm.momentum_status
    ORDER BY avg_churn_pct DESC
""")

if not alignment.empty:
    fig = px.bar(alignment, x='momentum_status', y='avg_churn_pct',
                 color='avg_churn_pct', color_continuous_scale='RdYlGn_r',
                 title='Average churn risk by spend momentum',
                 labels={'avg_churn_pct': 'Avg churn %', 'momentum_status': ''})
    fig.show()
    display(alignment)

## 6. Sample churn explanations
What do the actual explanations look like for the highest-risk customers?

In [ ]:
samples = q(f"""
    SELECT churn_risk_level,
           ROUND(churn_probability * 100, 1) AS churn_pct,
           ROUND(total_spend, 0) AS total_spend,
           days_since_last,
           reason_1, reason_2, reason_3
    FROM `{PROJECT}.marts.mart_churn_explained`
    WHERE churn_risk_level = 'Critical'
    ORDER BY total_spend DESC
    LIMIT 10
""")

if not samples.empty:
    print('Top 10 highest-spend Critical risk customers:\n')
    display(samples)
else:
    print('No Critical risk customers found.')